In [ ]:
!pip install requests pandas xlsxwriter


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.1/165.1 kB 3.1 MB/s eta 0:00:00


In [ ]:
import requests
import pandas as pd

# Lista krajów do analizy
countries = {
    'AUT': 'Austria', 'BEL': 'Belgium', 'BGR': 'Bulgaria', 'HRV': 'Croatia',
    'CYP': 'Cyprus', 'CZE': 'Czech Republic', 'DNK': 'Denmark', 'EST': 'Estonia',
    'FIN': 'Finland', 'FRA': 'France', 'DEU': 'Germany', 'GRC': 'Greece',
    'HUN': 'Hungary', 'IRL': 'Ireland', 'ITA': 'Italy', 'LVA': 'Latvia',
    'LTU': 'Lithuania', 'LUX': 'Luxembourg', 'MLT': 'Malta', 'NLD': 'Netherlands',
    'POL': 'Poland', 'PRT': 'Portugal', 'ROU': 'Romania', 'SVK': 'Slovakia',
    'SVN': 'Slovenia', 'ESP': 'Spain', 'SWE': 'Sweden'
}

# Wskaźnik eksportu jako % PKB
indicator = 'NE.EXP.GNFS.ZS'  # Export of goods and services as % of GDP
indicator_name = "Export as % of GDP"
start_year = 2004
end_year = 2024

# Funkcja do pobierania danych z API Banku Światowego
def fetch_data(country_code, indicator_code):
    url = f"https://api.worldbank.org/v2/country/{country_code}/indicator/{indicator_code}?date={start_year}:{end_year}&format=json"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        if len(data) > 1 and isinstance(data[1], list):
            return data[1]
    return []

# Pobieranie danych dla każdego kraju
data_list = []
for country_code, country_name in countries.items():
    data = fetch_data(country_code, indicator)
    for entry in data:
        year = entry.get('date')
        value = entry.get('value')
        if year and value is not None:
            data_list.append({
                'Year': int(year),
                'Country': country_name,
                indicator_name: value
            })

# Tworzenie DataFrame
df = pd.DataFrame(data_list)

# Pivotowanie danych (rok jako indeks, kraje jako kolumny)
df_pivot = df.pivot(index='Year', columns='Country', values=indicator_name)

# Tworzenie arkusza informacyjnego
info_data = {
    "Description": [
        "Source: World Bank Open Data",
        "Indicator: Export of Goods and Services (% of GDP)",
        "Definition: Exports of goods and services represent the value of all goods and other market services provided to the rest of the world.",
        f"Years Covered: {start_year}-{end_year}",
        "Countries Covered: " + ", ".join(countries.values())
    ]
}
df_info = pd.DataFrame(info_data)

# Zapis do pliku Excel z odpowiednimi arkuszami
output_file = "world_bank_export_gdp.xlsx"
with pd.ExcelWriter(output_file, engine="xlsxwriter") as writer:
    df_info.to_excel(writer, sheet_name="0 - Info", index=False)
    df_pivot.to_excel(writer, sheet_name="Export % of GDP")

print(f"Plik zapisano jako: {output_file}")


Plik zapisano jako: world_bank_export_gdp.xlsx
